In [ ]:
# Expectation to previous code

In [ ]:
#liabries installed:        !pip install chromadb Pillow matplotlib numpy
#imports:
                            #import chromadb
                            #import numpy as np
                            #from PIL import Image
                            #import matplotlib.pyplot as plt
                            #import os
                            #from sentence_transformers import SentenceTransformer
                            #model = SentenceTransformer('clip-ViT-B-32')

#instantiate the model:     client = chromadb.Client()

In [3]:
#Establish ChromaDB connection and load the client

In [ ]:
# Load environment variables from .env file

#load_dotenv() #we assume that a .env file has been created that stores that API credentials (shared in slack channel)

# Get the API key and other details from environment variables

#api_key = os.getenv("CHROMA_API_KEY")
#tenant_id = os.getenv("CHROMA_TENANT")
#database_name = os.getenv("CHROMA_DATABASE")

#load client
#client = chromadb.CloudClient(
        #api_key=api_key,
        #tenant=tenant_id,
        #database=database_name
    #)

In [2]:
import os
# TODO: update path to use os
import PIL.Image as Image

from sentence_transformers import SentenceTransformer

import chromadb
from PIL import Image

/home/gwenm/.pyenv/versions/3.10.6/envs/inspiart/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-10 12:02:25.833595: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [9]:
model = SentenceTransformer('clip-ViT-B-32')
chroma_client = chromadb.CloudClient(
        api_key=os.environ.get("CHROMA_API_KEY"),
        tenant=os.environ.get("CHROMA_TENANT"),
        database='inspiart'
        )
collection = chroma_client.get_or_create_collection(name="wikiart_115000images")

In [ ]:
def get_similar_images(image_query: Image.Image, collection, model, n_results: int = 5):
    """
    Finds the 5 most similar images in the collection for a given image query and
    handles the case where the query image itself is in the database.
    """
    # 1. Encode the input image to get its embedding
    query_embedding = model.encode(image_query, device="cpu").tolist()

    # 2. Query for 5 results + 1 to handle the case where the query image is the top match
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results + 1,
        include=['metadatas', 'distances']
    )

    # 3. Process the results
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]

    # Check if the most similar image is the image itself which means the distance is arbitrarilly close to 0
    is_exact_match = distances[0] < 0.000001

    # If it's an exact match, we return its metadata and the next 5 most similar images, otherwise, we just return the top 5.
    if is_exact_match:
        # The first result is the image itself, so we return its metadata.
        exact_match_data = {
            "artist": metadatas[0].get('artist'),
            "artwork_name": metadatas[0].get('document'),
        }
        similar_images_metadata = metadatas[1:]
    else:
        # The image is not in our dataset, so there is no exact match.
        exact_match_data = None
        similar_images_metadata = metadatas[:n_results]

   # 4. Format the list of similar images for the final output
    similar_images_output = []
    for meta in similar_images_metadata:
        similar_images_output.append({
            "artist": meta.get('artist'),
            "artwork_name": meta.get('document'),
        })

    # 5. Return a structured dictionary
    return {
        "query_image_found_in_db": is_exact_match,
        "exact_match_details": exact_match_data,
        "similar_images": similar_images_output
    }

In [14]:
path = "../raw_data/Images to try/Piet_Mondriaan,_1942_-_New_York_City_I.jpg"
img = Image.open(path)
get_similar_images(img, collection, model, n_results = 5)

{'query_image_found_in_db': False,
 'exact_match_details': None,
 'similar_images': [{'artist': 'Piet Mondrian', 'artwork_name': None},
  {'artist': 'Piet Mondrian', 'artwork_name': None},
  {'artist': 'Piet Mondrian', 'artwork_name': None},
  {'artist': 'Richard Paul Lohse', 'artwork_name': None},
  {'artist': 'Theo van Doesburg', 'artwork_name': None}]}

In [ ]:
def receive_image(path):

    #get the image from the POST request

    #contents = img.file.read()

    working_image = Image.open(path)

    #get or create a connection

    images_db = chroma_client.get_or_create_collection(name="wikiart_115000images")

    # Use the CLIP model to encode the image

    query_embedding = model.encode(working_image, device="cpu").tolist()

    #perform the query

    image_suggestions = images_db.query(
    query_embeddings=[query_embedding],
    include=['uris','metadatas', 'distances'],
    n_results=3
    )

    # Search if the image is matches with the first result as an output:
    distances = image_suggestions['distances'][0]

    print(image_suggestions['metadatas'][0])
    print(distances)

    return None
    is_exact_match = distances[0] < 0.000001

    # if it matches, return its infos, else return we don't know the image
    if is_exact_match:
        image_dict = {
            "artist" : image_suggestions['metadatas'][0][0]['artist'],
            "file_name" : image_suggestions['metadatas'][0][0]['file_name']
            }
    else :
        image_dict = {
            "artist" : "Unknown artist",
            "file_name" : "Unknown artwork"
            }

    return image_dict


In [29]:
path = "../raw_data/Images to try/Piet_Mondriaan,_1942_-_New_York_City_I.jpg"
receive_image(path)

[{'artist': 'Piet Mondrian', 'wikiart_url': 'https://www.wikiart.org/en/piet-mondrian/new-york-city-i-1942', 'file_name': '112511-new-york-city-i-1942.jpg', 'style': 'Neoplasticism', 'img_url': 'https://uploads5.wikiart.org/images/piet-mondrian/new-york-city-i-1942.jpg!Large.jpg', 'movement': 'Neo-Impressionism'}]
13.06532


In [34]:
path = "/home/gwenm/code/gwen-m97/inspiart/raw_data/wiki/new-york-city-i-1942.jpg!Large.jpg"
receive_image(path)

[{'style': 'Neoplasticism', 'file_name': '112511-new-york-city-i-1942.jpg', 'img_url': 'https://uploads5.wikiart.org/images/piet-mondrian/new-york-city-i-1942.jpg!Large.jpg', 'wikiart_url': 'https://www.wikiart.org/en/piet-mondrian/new-york-city-i-1942', 'artist': 'Piet Mondrian', 'movement': 'Neo-Impressionism'}]
5.4051226e-09


In [ ]:
path = "../raw_data/Images to try/Piet_Mondriaan,_1942_-_New_York_City_I.jpg"
receive_image(path)